# Examen Final – Sistema RAG sobre arXiv Paper Abstracts
**Desarrollado por:** Erick Romero

En este notebook se detalla el diseño, implementación y evaluación de un sistema de Recuperación Aumentada por Generación (RAG).

## Enlace al chat RAG: [Scientific RAG Assistant](https://scientific-rag-assistant.streamlit.app/)

## A. Preparación del corpus
Cargarmos la base de datos de Kaggle, seleccionaremos las columnas relevantes

In [ ]:
import kagglehub

# Descargar la última versión del dataset
path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

print("Path to dataset files:", path)

In [6]:
import os

os.listdir(path)

['arxiv_data.csv', 'arxiv_data_210930-054931.csv']

In [7]:
import pandas as pd
import os

csv_path = os.path.join(path, "arxiv_data.csv")

df = pd.read_csv(csv_path)
print(f"Número de documentos: {len(df):,}")
df.head()

Número de documentos: 51,774


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51774 entries, 0 to 51773
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titles     51774 non-null  str  
 1   summaries  51774 non-null  str  
 2   terms      51774 non-null  str  
dtypes: str(3)
memory usage: 1.2 MB


In [9]:
df.columns

Index(['titles', 'summaries', 'terms'], dtype='str')

### Limpieza de datos
Eliminamos registros nulos y duplicados, y normalizamos espacios y caracteres en los textos.

In [10]:
# Eliminacion de duplicados
duplicados = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicados eliminados: {duplicados}")

# Eliminacion de nulos
nulos_antes = df.isnull().sum().sum()
df = df.dropna()
print(f"Filas con nulos eliminadas: {nulos_antes}")

# Limpieza de espacios y caracteres especiales
for col in ['titles', 'summaries']:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    df[col] = df[col].str.replace(r'[^\w\s\.\,\-\:\;\\(\\)\[\]\/\?\!\'\']', '', regex=True)

# Comprobamos
print(f"\nDocumentos después de limpieza: {len(df):,}")
print(f"\nNulos por columna:\n{df.isnull().sum()}")
print(f"\nDuplicados restantes: {df.duplicated().sum()}")
print(f"\nNúmero de documentos finales: {len(df):,}")
df.head()

Duplicados eliminados: 12783
Filas con nulos eliminadas: 0

Documentos después de limpieza: 38,991

Nulos por columna:
titles       0
summaries    0
terms        0
dtype: int64

Duplicados restantes: 0

Número de documentos finales: 38,991


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


In [11]:
df["document"] = (
    "Title: " + df["titles"] +
    "\n\nSummaries: " + df["summaries"]
)

In [12]:
df.head()

,titles,summaries,terms,document
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']",Title: Survey on Semantic Stereo Matching / Se...
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']",Title: FUTURE-AI: Guiding Principles and Conse...
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']",Title: Enforcing Mutual Consistency of Hard Re...
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV'],Title: Parameter Decoupling Strategy for Semi-...
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']",Title: Background-Foreground Segmentation for ...


# B. Representación mediante Embeddings

Para representar semánticamente los documentos del corpus se empleó un modelo de **Sentence Transformers**.

Se seleccionó el modelo **BAAI/bge-small-en-v1.5**, ya que ofrece un buen equilibrio entre calidad, velocidad y tamaño del modelo. Además, está optimizado para tareas de búsqueda semántica (semantic search), siendo ampliamente utilizado en sistemas RAG.

Cada documento (compuesto por el título y el abstract) es transformado en un vector denso de alta dimensión. Estos vectores permiten medir la similitud semántica entre consultas y documentos utilizando métricas como la similitud del coseno.

In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5062.42it/s]


In [ ]:
# Calcula los embeddings y los guarda
embeddings = embedding_model.encode(
    df["document"].tolist(),
    batch_size=120,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
# Guardar embeddings y corpus limpio
np.save('data/arxiv_embeddings.npy', embeddings)
df.to_csv('arxiv_cleaned.csv', index=False)
print(f'Embeddings creados: {embeddings.shape}. Guardados en arxiv_embeddings.npy')

In [14]:
# embeddings = np.load('arxiv_embeddings.npy')

In [15]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(38991, 384)


# C. Almacenamiento y búsqueda vectorial

Los embeddings generados en la etapa anterior se almacenan en una base de datos vectorial utilizando ****.

Cada documento se almacena junto con:

- Un identificador único.
- El embedding correspondiente.
- El texto completo del documento.
- Metadatos (título y categoría).

Esto permite realizar búsquedas semánticas eficientes mediante la comparación entre el embedding de una consulta y los embeddings almacenados.

In [16]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

38991


In [17]:
faiss.write_index(
    index,
    "arxiv_index.faiss"
)

Cada embedding en el índice FAISS mantiene el mismo orden que el DataFrame `df`. Esto nos permite, al recuperar los índices de los vecinos más cercanos, acceder directamente a los documentos correspondientes.

# D. Recuperación

Para recuperar documentos relevantes se implementa un enfoque híbrido:

1. Recuperación inicial mediante búsqueda vectorial con FAISS.
2. Re-ranking mediante un modelo Cross Encoder.

FAISS permite encontrar rápidamente los documentos cuyos embeddings tienen mayor similitud semántica con la consulta.

Posteriormente, el Cross Encoder evalúa directamente la relación entre la consulta y cada documento recuperado, mejorando la precisión de los resultados.

In [18]:
# Función de búsqueda semántica
def buscar(query, k=5):
    # 1. Generar embedding de la consulta (normalizado)
    q_emb = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    # 2. Buscar los k vecinos más cercanos
    scores, indices = index.search(q_emb, k)
    # 3. Recuperar documentos del DataFrame
    resultados = df.iloc[indices[0]].copy()
    resultados["score"] = scores[0]
    return resultados[["titles", "terms", "score", "document"]]

In [19]:
# Ejemplo de busqueda
query_ejemplo = "What are the main applications of Graph Neural Networks?"
resultados = buscar(query_ejemplo, k=25)
print(f"Consulta: {query_ejemplo}\n")
resultados

Consulta: What are the main applications of Graph Neural Networks?



,titles,terms,score,document
19175,"Graph Neural Networks: Methods, Applications, ...","['cs.LG', 'cs.AI', '68Txx', 'I.2.6; I.2; I.5']",0.836709,"Title: Graph Neural Networks: Methods, Applica..."
3066,On Node Features for Graph Neural Networks,"['cs.LG', 'stat.ML']",0.819328,Title: On Node Features for Graph Neural Netwo...
22725,"Graph Neural Networks: Architectures, Stabilit...","['cs.LG', 'stat.ML']",0.816301,"Title: Graph Neural Networks: Architectures, S..."
19977,A Comprehensive Survey on Graph Neural Networks,"['cs.LG', 'stat.ML']",0.815398,Title: A Comprehensive Survey on Graph Neural ...
23256,A Practical Guide to Graph Neural Networks,"['cs.LG', 'cs.AI', 'cs.SI']",0.814621,Title: A Practical Guide to Graph Neural Netwo...
21215,Computing Graph Neural Networks: A Survey from...,"['cs.LG', 'cs.DC', 'stat.ML']",0.807436,Title: Computing Graph Neural Networks: A Surv...
22456,"Should Graph Neural Networks Use Features, Edg...",['cs.LG'],0.805296,Title: Should Graph Neural Networks Use Featur...
21895,"Graphs, Convolutions, and Neural Networks: Fro...","['cs.LG', 'cs.SY', 'eess.SY', 'stat.ML']",0.804084,"Title: Graphs, Convolutions, and Neural Networ..."
2772,A Gentle Introduction to Deep Learning for Graphs,"['cs.LG', 'cs.SI', 'stat.ML']",0.803345,Title: A Gentle Introduction to Deep Learning ...
3223,Graph Neural Networks for Small Graph and Gian...,"['cs.LG', 'cs.NE', 'stat.ML']",0.803297,Title: Graph Neural Networks for Small Graph a...


In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4686.57it/s]


In [21]:
pairs = [
    (
        query_ejemplo,
        row["document"]
    )
    for _, row in resultados.iterrows()
]

rerank_scores = reranker.predict(pairs)

In [22]:
resultados["rerank_score"] = rerank_scores

resultados = resultados.sort_values(
    "rerank_score",
    ascending=False
)

In [23]:
resultados.head(5)

,titles,terms,score,document,rerank_score
22211,Graph Neural Networks: A Review of Methods and...,"['cs.LG', 'cs.AI', 'stat.ML']",0.798615,Title: Graph Neural Networks: A Review of Meth...,6.808418
22703,Learning Graph Representations,"['cs.LG', 'cs.SI']",0.796004,Title: Learning Graph Representations\n\nSumma...,6.431347
19977,A Comprehensive Survey on Graph Neural Networks,"['cs.LG', 'stat.ML']",0.815398,Title: A Comprehensive Survey on Graph Neural ...,5.490592
3223,Graph Neural Networks for Small Graph and Gian...,"['cs.LG', 'cs.NE', 'stat.ML']",0.803297,Title: Graph Neural Networks for Small Graph a...,5.388867
19175,"Graph Neural Networks: Methods, Applications, ...","['cs.LG', 'cs.AI', '68Txx', 'I.2.6; I.2; I.5']",0.836709,"Title: Graph Neural Networks: Methods, Applica...",5.334636


# E. Generación aumentada por recuperación (RAG)

Para generar respuestas se implementa un pipeline RAG.

Primero se recuperan los documentos más relevantes mediante búsqueda semántica y posteriormente estos documentos son utilizados como contexto para un Modelo de Lenguaje Grande (LLM).

El modelo generativo recibe:

- La consulta del usuario.
- Los documentos recuperados.
- Instrucciones para responder únicamente utilizando la información disponible.

Si los documentos recuperados no contienen información suficiente, el sistema debe indicarlo explícitamente.

In [82]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()

llm_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [96]:
def rag_pipeline(query, top_k=5):
    retrieved = buscar(query, k=25)
    pairs = [(query, row['document']) for _, row in retrieved.iterrows()]
    rerank_scores = reranker.predict(pairs)
    retrieved['rerank_score'] = rerank_scores
    retrieved = retrieved.sort_values('rerank_score', ascending=False).head(top_k)

    if retrieved['score'].max() < 0.50:
        msg = 'The available documents do not contain enough information to answer this question.'
        return msg, retrieved

    contexto = '\n\n---\n\n'.join(
        f'Document {i+1}:\n{row["document"]}'
        for i, (_, row) in enumerate(retrieved.iterrows())
    )

    prompt = f"""You are a scientific assistant. Answer the question using ONLY the information provided in the context below. Do not use prior knowledge. If the context does not contain enough information, say: "The available documents do not contain enough information to answer this question."

Context:
{contexto}

Question: {query}

Answer:"""

    response = llm_client.chat.completions.create(
        model='openai/gpt-oss-20b:free',
        messages=[
            {'role': 'system', 'content': 'You answer scientific questions based solely on the provided context, without adding external knowledge.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content, retrieved

In [98]:
respuesta, retrieved_docs = rag_pipeline('What are the main applications of Graph Neural Networks?')
print('=== RESPUESTA GENERADA ===')
print(respuesta)
print('\n=== EVIDENCIAS (top 3) ===')
retrieved_docs.head(3)[['titles', 'score', 'rerank_score']]

=== RESPUESTA GENERADA ===
**Main applications of Graph Neural Networks (GNNs)**  

| Application | Typical tasks / domains mentioned |
|-------------|-----------------------------------|
| **Node‑level prediction** | Node classification (e.g., classifying entities in a network) |
| **Graph‑level prediction** | Graph classification (e.g., classifying whole molecules or sub‑graphs) |
| **Link prediction** | Predicting missing or future edges in a graph |
| **Knowledge‑graph representation** | Learning low‑dimensional embeddings for large, dynamic knowledge graphs |
| **Physical system modeling** | Modeling dynamics of physics systems |
| **Molecular fingerprint learning** | Generating fingerprints for molecules |
| **Protein interface prediction** | Predicting interactions between proteins |
| **Disease classification** | Classifying diseases based on graph‑structured biomedical data |
| **Structure reasoning in other domains** | Reasoning on extracted structures such as dependency tree

,titles,score,rerank_score
22211,Graph Neural Networks: A Review of Methods and...,0.798615,6.808418
22703,Learning Graph Representations,0.796004,6.431347
19977,A Comprehensive Survey on Graph Neural Networks,0.815398,5.490592


In [97]:
respuesta, retrieved_docs = rag_pipeline('Who is the best football player in the world?') # MESSI
print('=== RESPUESTA GENERADA ===')
print(respuesta)

=== RESPUESTA GENERADA ===
The available documents do not contain enough information to answer this question.
